In [1]:
import torch
print(f"CUDA disponible : {torch.cuda.is_available()}")
print(f"Device : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

CUDA disponible : True
Device : Tesla T4


In [2]:
# Clone du repo
!git clone https://github.com/Arcsin720/Sport-News.git
%cd /content/Sport-News

Cloning into 'Sport-News'...
remote: Enumerating objects: 24, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 24 (delta 1), reused 7 (delta 1), pack-reused 15 (from 2)
Receiving objects: 100% (24/24), 54.25 MiB | 25.80 MiB/s, done.
Resolving deltas: 100% (1/1), done.
/content/Sport-News


In [3]:

import torch
print(f"transformers version par défaut de Colab")
import transformers
print(f"transformers : {transformers.__version__}")
print(f"CUDA : {torch.cuda.is_available()} | {torch.cuda.get_device_name(0)}")

transformers version par défaut de Colab
transformers : 5.9.0
CUDA : True | Tesla T4


In [4]:
from datasets import load_dataset
import pandas as pd

SPORT_TOPICS = [
    "athletisme", "basket", "blog-du-tour-de-france", "blog-roland-garros",
    "championnats-monde-athletisme", "coupe-du-monde", "coupe-du-monde-rugby",
    "cyclisme", "football", "formule-1", "golf", "handball",
    "jeux-olympiques", "jeux-olympiques-pyeongchang-2018", "jeux-olympiques-rio-2016",
    "le-nouveau-roland-garros", "ligue-1", "ligue-des-champions",
    "natation", "roland-garros", "rugby", "ski", "sport",
    "tennis", "top-14", "tour-de-france", "voile",
]

BASE_URL = "hf://datasets/reciTAL/mlsum@refs/convert/parquet/fr"
dataset_test = load_dataset("parquet", data_files={"test": f"{BASE_URL}/test/*.parquet"})

df_test = dataset_test["test"].to_pandas()
df_sport_test = df_test[df_test["topic"].isin(SPORT_TOPICS)].reset_index(drop=True)

print(f" Test set sport : {len(df_sport_test)} articles")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


fr/test/0000.parquet:   0%|          | 0.00/42.6M [00:00<?, ?B/s]

Generating test split: 0 examples [00:00, ? examples/s]

✅ Test set sport : 1097 articles


In [5]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch, time

MODEL_NAME = "csebuetnlp/mT5_multilingual_XLSum"

print(f"Chargement de {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to("cuda")
model.eval()
print(" Modèle chargé")

# Test sur 3 articles
samples = df_sport_test.sample(n=3, random_state=42).reset_index(drop=True)

for i, row in samples.iterrows():
    inputs = tokenizer(row["text"], max_length=512, truncation=True, return_tensors="pt").to("cuda")
    t0 = time.time()
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_length=64,
            num_beams=4,
            no_repeat_ngram_size=2,
            early_stopping=True,
        )
    summary = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    duration = time.time() - t0

    print(f"\n[{i+1}] {row['topic']} ({duration:.1f}s)")
    print(f"   Réf : {row['summary']}")
    print(f"   mT5 : {summary}")

Chargement de csebuetnlp/mT5_multilingual_XLSum...


config.json:   0%|          | 0.00/730 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/375 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.33G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.33G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


✅ Modèle chargé

[1] sport (6.6s)
  📝 Réf : Le Suédois a présidé pendant dix-sept ans, de 1990 à 2007, l’UEFA, avant d’être battu par Michel Platini. Il fut nommé dans la foulée président honoraire de l’UEFA.
  🤖 mT5 : Lennart Johansson est mort à l’âge de 89 ans, a annoncé mercredi 5 juin la Fédération suédoise de football.

[2] sport (1.9s)
  📝 Réf : Le sélectionneur Jacques Brunel a annoncé mardi la composition de l’équipe de France qui affrontera l’Ecosse, samedi à Edimbourg. Le capitaine Guirado fait son retour.
  🤖 mT5 : Après la victoire de l’équipe de France, le sélectionneur Jacques Brunel a dévoilé la composition du XV deFrance appelé à disputer le second match de préparation aux éliminatoires de la Coupe du Monde.

[3] football (1.6s)
  📝 Réf : Malgré une nette évolution, la pratique féminine du football souffre encore de stéréotypes sexistes. Organisé en France du 7 juin au 7 juillet, le Mondial est l’occasion d’y remédier. Plus qu’un tournoi, une reconnaissance à gagner.
 

In [6]:
%cd /content/Sport-News
!git pull

/content/Sport-News
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 4 (delta 2), reused 4 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 1019 bytes | 254.00 KiB/s, done.
From https://github.com/Arcsin720/Sport-News
   92b0fa6..3475759  main       -> origin/main
Updating 92b0fa6..3475759
Fast-forward
 src/summarizer.py | 84 ++++++++++++++++++++-----------------------------------
 1 file changed, 30 insertions(+), 54 deletions(-)


In [7]:
# on recharge le module mis à jour
import importlib
import src.summarizer
importlib.reload(src.summarizer)
from src.summarizer import LLMSummarizer

# Chargement
llm = LLMSummarizer()

# Test sur les mêmes  articles
import time

print("\n" + "-" * 80)
print("RÉSUMÉS CroissantLLM (via prompting)")


for i, row in samples.iterrows():
    t0 = time.time()
    summary = llm.summarize(row["text"])
    duration = time.time() - t0

    print(f"\n[{i+1}] {row['topic']} ({duration:.1f}s)")
    print(f"   Référence    : {row['summary']}")
    print(f"   CroissantLLM : {summary}")

Chargement du LLM croissantllm/CroissantLLMChat-v0.1 sur cuda...


config.json:   0%|          | 0.00/732 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/19.2k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.35M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/18.0k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

LLM chargé

RÉSUMÉS CroissantLLM (via prompting)

[1] sport (4.7s)
  📝 Référence    : Le Suédois a présidé pendant dix-sept ans, de 1990 à 2007, l’UEFA, avant d’être battu par Michel Platini. Il fut nommé dans la foulée président honoraire de l’UEFA.
  🤖 CroissantLLM : Lennart Johansson, ancien président de l'Union européenne du football (UEFA) de 1990 à 2007, est décédé à l'âge de 89 ans. Il s'est endormi dans la soirée du 4 juin à la suite d'une courte maladie. Né dans une famille d'ouvriers de Bromma, un quartier ouest de Stockholm, coursier à 15 ans dans une société de travaux publics dont il dirigera ensuite le conseil d'administration, Lennart Johansson avait commencé sa carrière de dirigeant sportif dans les années soixante, dans le handball. Avant d'entamer une carrière dans le football et d'être élu en 1967 à la présidence de l'AIK, club de

[2] sport (4.2s)
  📝 Référence    : Le sélectionneur Jacques Brunel a annoncé mardi la composition de l’équipe de France qui affrontera l